# Camada Silver — Tratamento dos Dados

Nesta etapa será realizado o tratamento da base `history_bronze.csv`, gerada na camada Bronze.

O objetivo é verificar a qualidade dos dados, identificar possíveis inconsistências, tratar valores ausentes e ajustar os tipos de dados quando necessário.

Os tratamentos serão realizados após a inspeção da base, mantendo os dados originais preservados na camada Bronze.

In [6]:
import pandas as pd
from pathlib import Path

In [7]:
pasta_bronze = Path("dados/bronze")
pasta_silver = Path("dados/silver")

## Leitura da base Bronze

A base `history_bronze.csv` será carregada a partir da camada Bronze.

Neste momento, os dados serão apenas importados para o ambiente de trabalho, sem aplicação de tratamentos ou alterações em seu conteúdo.

In [9]:
arquivo_bronze = pasta_bronze / "history_bronze.csv"

df_silver = pd.read_csv(arquivo_bronze)

df_silver.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome
0,4578-PHJYZ,Male,0,Yes,Yes,52,Yes,No,DSL,No,...,Yes,No,One year,Yes,Electronic check,68.75,3482.85,No,I have been a customer with this internet prov...,6532
1,6289-CPNLD,Male,0,Yes,Yes,33,Yes,No,DSL,No,...,Yes,Yes,One year,Yes,Mailed check,73.90,2405.05,Yes,I've been a customer with this company for ove...,7634
2,2682-KEVRP,Female,1,No,No,22,Yes,No,No,No internet service,...,No internet service,No internet service,One year,Yes,Mailed check,20.05,417.00,No,I have been a customer with this company for 2...,3628
3,5697-GOMBF,Female,1,Yes,Yes,28,No,No phone service,DSL,No,...,Yes,No,Month-to-month,Yes,Electronic check,35.90,973.65,No,I have been a customer with this internet prov...,7851
4,9717-QEBGU,Male,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.30,144.00,No,I have been using the fiber optic internet ser...,1691


## Inspeção da estrutura dos dados

Após a leitura da base Bronze, será realizada uma inspeção inicial para verificar a quantidade de registros e variáveis disponíveis.

Essa verificação permitirá confirmar se a estrutura dos dados foi mantida antes do início dos tratamentos.

In [10]:
df_silver.shape

(5634, 23)

### Dimensões da base

A base contém **5.634 registros e 23 variáveis**, mantendo as mesmas dimensões identificadas na camada Bronze.

Com as dimensões confirmadas, a próxima etapa será verificar os tipos de dados de cada variável.

In [11]:
df_silver.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                  str
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
CustomerFeedback        str
MonthlyIncome         int64
dtype: object

### Análise dos tipos de dados

A inspeção mostrou que a maior parte das variáveis apresenta tipos compatíveis com seu conteúdo.

Entretanto, a variável `tenure`, que representa o tempo de permanência dos clientes, foi identificada como texto (`str`).

Antes de realizar qualquer alteração, será necessário verificar os valores presentes nessa coluna.

In [13]:
df_silver["tenure"].unique()

<StringArray>
[     '52',      '33',      '22',      '28',       '2',      '70',       '3',
      '15',      '42',      '25',      '23',       '7',       '1',      '54',
      '30',      '67',      '72',      '58',      '71',      '40',      '60',
      '36',      '10',       '9',      '53',      '61',      '49',      '47',
      '41',      '29',      '66',      '24',      '69',      '11',      '34',
      '59',      '51',      '64',       '4',      '39',      '12',      '20',
      '14',      '68',      '44',      '35',      '43',      '19',      '13',
      '37',      '63',      '26',      '55',       '6',      '46',      '50',
       '5',      '17',      '27',      '32',       '8',      '21',      '31',
      '45',      '65',      '56',      '57',      '62',      '16',      '18',
      '38',      '48', 'unknown',       '0']
Length: 74, dtype: str

### Identificação de valores inconsistentes em tenure

A inspeção identificou valores numéricos armazenados como texto e a presença do valor `unknown` na variável `tenure`.

Como essa variável representa o tempo de permanência dos clientes, será verificada a quantidade de registros com valores desconhecidos antes de definir o tratamento adequado.

In [14]:
df_silver["tenure"].value_counts()

tenure
1          475
72         266
2          188
3          161
4          144
          ... 
44          41
36          39
39          37
unknown     19
0           10
Name: count, Length: 74, dtype: int64

### Análise dos registros com tenure desconhecido

Foram identificados 19 registros com o valor `unknown` na variável `tenure`, representando aproximadamente 0,34% da base.

Antes de realizar o tratamento, esses registros serão analisados para verificar suas características e avaliar a melhor forma de lidar com os valores desconhecidos.

In [15]:
df_silver[df_silver["tenure"] == "unknown"]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome
1059,7301-ABVAD,Female,0,No,No,unknown,Yes,Yes,DSL,No,...,Yes,Yes,Two year,No,Bank transfer (automatic),86.60,6350.50,No,I have been a customer with this company for 6...,3093
1097,9651-GTSAQ,Female,0,Yes,No,unknown,Yes,No,Fiber optic,Yes,...,Yes,Yes,One year,Yes,Bank transfer (automatic),106.10,NaN,Yes,I have been a customer with this company for o...,4702
1791,0135-NMXAP,Female,0,No,No,unknown,Yes,Yes,Fiber optic,No,...,Yes,No,Month-to-month,Yes,Bank transfer (automatic),89.75,1052.40,Yes,I recently cancelled my service with this comp...,7709
1984,3892-NXAZG,Male,0,Yes,Yes,unknown,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,Two year,Yes,Credit card (automatic),114.05,8289.20,No,I have been a customer with this company for 6...,4415
2361,3934-HXCFZ,Male,0,Yes,No,unknown,Yes,Yes,Fiber optic,Yes,...,Yes,No,Month-to-month,Yes,Electronic check,99.25,2186.40,Yes,I have been a customer of this fiber optic int...,9165
2372,1622-HSHSF,Female,0,No,No,unknown,Yes,No,No,No internet service,...,No internet service,No internet service,Month-to-month,Yes,Mailed check,19.55,19.55,No,I have only been a customer with this company ...,4751
2641,6907-NZZIJ,Female,0,No,No,unknown,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,99.15,3615.60,Yes,I have been a customer with this company for 3...,4701
2896,6023-GSSXW,Female,0,Yes,No,unknown,Yes,No,DSL,No,...,No,Yes,Two year,Yes,Credit card (automatic),61.45,4131.20,No,I have been a customer with this company for a...,2494
2936,5688-KZTSN,Male,0,Yes,Yes,unknown,Yes,No,No,No internet service,...,No internet service,No internet service,Month-to-month,Yes,Electronic check,20.00,288.05,Yes,I recently cancelled my service with this prov...,4749
2959,7694-VLBWQ,Male,0,Yes,No,unknown,Yes,Yes,Fiber optic,No,...,Yes,Yes,One year,No,Electronic check,104.10,7040.85,Yes,I have been a customer with this company for o...,8867


### Investigação dos valores desconhecidos em tenure

A análise identificou 19 registros com tempo de permanência desconhecido.

Como a maioria desses clientes possui valores acumulados de cobrança, será verificada a possibilidade de estimar o tempo de permanência a partir das informações financeiras disponíveis.

Essa análise será utilizada para avaliar uma possível estratégia de tratamento, sem alterar os dados neste momento.

In [16]:
df_silver.loc[
    df_silver["tenure"] == "unknown",
    ["customerID", "MonthlyCharges", "TotalCharges"]
].assign(
    tenure_estimado=lambda df: (
        df["TotalCharges"] / df["MonthlyCharges"]
    ).round()
)

,customerID,MonthlyCharges,TotalCharges,tenure_estimado
1059,7301-ABVAD,86.60,6350.50,73.0
1097,9651-GTSAQ,106.10,NaN,NaN
1791,0135-NMXAP,89.75,1052.40,12.0
1984,3892-NXAZG,114.05,8289.20,73.0
2361,3934-HXCFZ,99.25,2186.40,22.0
2372,1622-HSHSF,19.55,19.55,1.0
2641,6907-NZZIJ,99.15,3615.60,36.0
2896,6023-GSSXW,61.45,4131.20,67.0
2936,5688-KZTSN,20.00,288.05,14.0
2959,7694-VLBWQ,104.10,7040.85,68.0


### Verificação de valores ausentes

Nesta etapa será verificada a quantidade de valores ausentes em cada coluna da base.

O objetivo é identificar quais variáveis precisam de tratamento antes de prosseguir com a padronização dos dados.

In [17]:
df_silver.isnull().sum()

customerID            0
gender                0
SeniorCitizen         0
Partner               0
Dependents            0
tenure                0
PhoneService          0
MultipleLines         0
InternetService       0
OnlineSecurity        0
OnlineBackup          0
DeviceProtection      0
TechSupport           0
StreamingTV           0
StreamingMovies       0
Contract              0
PaperlessBilling      0
PaymentMethod         0
MonthlyCharges        0
TotalCharges        278
Churn                 0
CustomerFeedback      0
MonthlyIncome         0
dtype: int64